<a href="https://colab.research.google.com/github/SouvickC/time_series_prediction_project/blob/main/2_BaselineModel%5CTime_Series_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, log_loss

In [ ]:
# !pip install sktime[all_extras] -qq

In [21]:
DATASET_PATH = Path("/content/drive/MyDrive/Colab Notebooks/tdcsfog")
TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"

In [22]:
def load_file(file_path: Path) -> tuple[np.ndarray, float]:

    # Load the data into a numpy ndarray
    with open(file_path, "rb") as infile:
        arr = np.load(infile)

    # Extract the accelerometer data as the input features
    features = arr[:, 1:4]

    # Extract the labels
    labels = arr[:, 4:]
    labels = np.max(labels, axis=-1)
    return features, np.any(labels).astype(float)

In [42]:
loaded_arr = np.load("/content/drive/MyDrive/Colab Notebooks/tdcsfog/train/003f117e14_0000.npy")
print(loaded_arr[0])

dftrain = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/tdcsfog/train.csv')
dftrain.head()

[ 0.         -9.5339393   0.56632163 -1.41352531  0.          0.
  0.        ]


,file_name,label
0,76c7edf878_0004.npy,1.0
1,e7b5afe544_0014.npy,1.0
2,76c7edf878_0011.npy,1.0
3,2a9faf5644_0000.npy,0.0
4,5a70242f37_0002.npy,1.0


In [93]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((4136, 3, 1280), (4136,), (960, 3, 1280), (960,))

In [114]:
X_list = []
y_list = []

for fpath in tqdm(TRAIN_PATH.iterdir(), desc="Loading all rows from all files"):
    if not fpath.is_file() or fpath.suffix != ".npy":
        continue

    data = np.load(fpath)              # shape: (N, D), e.g., (1280, 6)
    X_list.append(data[:, 1:4])        # 3 feature columns per row, shape: (N, 3)

    labels = np.max(data[:, 4:], axis=1)  # e.g., max across label columns per row, shape: (N,)
    y_list.append(labels)

# Now concatenate all rows
X_train = np.vstack(X_list)           # shape: (total_rows, 3)
y_train = np.hstack(y_list)           # shape: (total_rows,)

print("Final shapes:", X_train.shape, y_train.shape)

Loading all rows from all files: 4136it [01:42, 40.46it/s] 


Final shapes: (5294080, 3) (5294080,)


array([1., 1., 0., ..., 0., 0., 0.])

In [104]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

ValueError: Found array with dim 3. RandomForestClassifier expected <= 2.